# Análisis de Datos Proteómicos Públicos

## Introducción

En esta práctica analizamos datos reales de proteómica obtenidos mediante **espectrometría de masas (MS)** y depositados en el repositorio público **PRIDE Archive**.  
El objetivo general es comprender cómo se identifican y cuantifican péptidos y proteínas, y cómo podemos interpretar estos datos para extraer información biológica.

---

## Proyecto utilizado: PXD068675 — *Supplementary Dataset S6. AvrRpm1 ADP-ribosylates PLDGAMMA3 when expressed in N. benthamiana*


El archivo elegido para esta práctica proviene del proyecto **PXD068675**, disponible en la ruta oficial del repositorio PRIDE:

-> **https://www.ebi.ac.uk/pride/archive/projects/PXD068675**


Para los ejercicios de esta práctica utilizamos el archivo:

- **`peptides.txt`**

Este archivo contiene la información necesaria para trabajar a nivel de **péptidos**: secuencias, proteínas asociadas, genes, valores de confianza (PEP) e intensidades en distintas muestras.

En los ejercicios eliminamos aquella información que no nos resulta útil para que sea más sencillo trabajar con él.

---

## Indicaciones importantes para hacer los ejercicios dadas por la profesora

> En casos ambiguos, **no se puede determinar con certeza** a qué grupo pertenece un péptido o proteína.  
> Si la concentración es **anormalmente alta**, debe **justificarse** claramente (si se argumenta bien, **sí se valora positivamente**).  
> La ausencia de señal **no implica necesariamente** que la proteína esté presente o ausente.  
> Se deben **explicar correctamente los p-values**:  
> - Qué significan,  
> - Cómo interpretarlos,  
> - Por qué apoyan (o no) una diferencia entre grupos.  
> Las justificaciones deben ser **breves**, preferiblemente **en una sola línea**, salvo casos muy excepcionales.
---

#### **VAMOS A QUEDARNOS CON AQUELLA INFORMACIÓN RELEVANTE Y NECESARIA PARA LOS EJERCICIOS ÚNICAMENTE:**

In [ ]:
import pandas as pd 

peptides = pd.read_csv("peptides.txt", sep="\t")

print(peptides.head())
print(peptides.columns)


### Filtrado de los datos para solo tomar las columnas necesarias:

## Ejercicios

### **Ejercicio 1 — Identificando péptidos**

**Tarea:**  
Selecciona un péptido de la tabla y responde:

- ¿Cuál es su secuencia (**Sequence**)?
- ¿A qué proteína(s) se asocia (**Proteins**)?
- ¿Cuál es el nombre del gen asociado (**Gene names**)?

**Objetivo:**  

Comprender que en proteómica **no se identifican proteínas directamente**, sino **péptidos**, que luego se asignan a las proteínas.

---



#### Para mayor comidad, lo pasamos a formato .csv para hacer uso e funciones que facilitan el trabajo.

In [ ]:
peptides_filtered.to_csv("peptides_filtered.csv", index=False)

#### ***Ya que nos interesa una secuencia, tomamos, por ejemplo la primera cadena de péptidos:***

In [ ]:
peptides = pd.read_csv("peptides_filtered.csv")

In [ ]:
first_peptide = peptides.iloc[0]

In [ ]:
sequence = first_peptide["Peptide.sequence"]
proteins = first_peptide["Protein.s."]
gene_name = first_peptide["Gene.name.s."]

### Resultados ejercicio 1:

In [ ]:
print("=== R E S U L T A D O S ===")
print(f"- Secuencia: {sequence}")
print(f"- Proteína(s) asociada(s): {proteins}")
print(f"- Nombre del gen asociado: {gene_name}")

### **Ejercicio 2 — Evaluando la confianza en la identificación**

**Tarea:**  
Observa el valor **PEP** del péptido seleccionado.

Interpretación:

- **PEP < 0.01** → identificación **confiable**  
- **PEP > 0.05** → identificación **poco confiable**

**Objetivo:**  

Comprender que la identificación de péptidos se basa en **probabilidades**, no en certezas absolutas.  
PEP indica la probabilidad de que la identificación sea incorrecta.

---

#### Seguimos usando la primera secuencia de péptidos que tomamos en el apartado anterior:

#### Además, tenemos varias columnas con diferentes valores PEP porque hay varios sets de búsqueda. En este caso, nos quedamos solo con la evaluación más pequeña obtenida, es decir, el PEP mínimo.

In [ ]:
pep_cols = [c for c in peptides.columns if c.endswith("_PEP")]

In [ ]:
pep_values = first_peptide[pep_cols]

In [ ]:
for col in pep_cols:
    raw_value = first_peptide[col]
    float_value = float(raw_value)
    print(f"  {col}: {raw_value}   →   {float_value:.6f}")

#### Observamos que todos los valores PEP asociados a este péptido son extremadamente pequeños. Al convertirse a formato numérico de coma flotante, estos valores son tan cercanos a cero que se redondean a 0.0.

In [ ]:
pep_values_float = first_peptide[pep_cols].astype(float)
min_pep = pep_values_float.min()
print(f'Valor PEP mínimo:', min_pep)

In [ ]:
if min_pep < 0.01:
    interpretation = "Identificación CONFIABLE (PEP < 0.01)"
elif min_pep > 0.05:
    interpretation = "Identificación POCO CONFIABLE (PEP > 0.05)"
else:
    interpretation = "Identificación con confianza intermedia (0.01 ≤ PEP ≤ 0.05)"


In [ ]:
print(interpretation)

### **Ejercicio 3 — Explorando la cuantificación entre muestras**

**Tarea:**  
Compara las intensidades del mismo péptido entre dos muestras (por ejemplo, **A1** y **A10**).

Preguntas guía:

- ¿Dónde es más abundante?
- ¿Está ausente (0 o vacío) en alguna muestra?

**Objetivo:**  
Interpretar la cuantificación experimental y comprender que puede variar por **factores biológicos o técnicos**.

---

In [ ]:
intensity_cols = [c for c in peptides.columns if "intensity" in c.lower()]

In [ ]:
pd.set_option("display.max_rows", None)
print(intensity_cols)

### **Ejercicio 4 — Identificando valores faltantes**

**Tarea:**  
Selecciona uno o dos péptidos que presenten valores faltantes (**0** o vacío) en alguna muestra.

Reflexiona:

- ¿Se debe a una verdadera ausencia biológica?
- ¿O a límites de detección del instrumento?

**Objetivo:**  
Comprender el concepto de valores faltantes **MNAR** (Missing Not At Random), típico en proteómica.

---

### **Ejercicio 5 — Comparación basada en proteínas**

**Tarea:**

1. Elige un gen o proteína (desde **Gene names**).  
2. Identifica todos los péptidos asociados a esa proteína.  
3. Compara las intensidades entre dos grupos de muestras (A vs B).

**Objetivo:**  
Comprender que la cuantificación proteica se basa en **varios péptidos**, no en uno solo.

---

### **Ejercicio 6 — Pregunta de razonamiento**

**Tarea:**  
Explica por qué es importante **identificar más de un péptido por proteína**.

---

### **Ejercicio 7 — Comparación estadística entre dos grupos (en R)**

**Tarea:**  
Determina si existe una diferencia significativa en la **abundancia media** de los péptidos entre dos grupos, por ejemplo:

- Grupo A1–A5  
- Grupo A6–A10

**Objetivo:**  
Aplicar una prueba estadística para evaluar si existe una diferencia real entre grupos experimentales o biológicos.

---